# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI(base_url = "http://localhost:11434/v1/", api_key = "ollama")

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gemma4:e4b", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
todos, completed = [], []
loop(messages)

Todo #1: Identify the distance between Boston and New York. Since it is not given, I must estimate a reasonable 
value for this distance.
Todo #2: Set up equations describing the position of each train as a function of time.
Todo #3: Account for the one-hour head start of the first train.
Todo #4: Solve the resulting equation for the time they meet.

A reasonable estimate for the distance between Boston and New York is approximately 200 miles. I will use this 
value for the calculation.

Todo #1: Identify the distance between Boston and New York. Since it is not given, I must estimate a reasonable 
value for this distance.
Todo #2: Set up equations describing the position of each train as a function of time.
Todo #3: Account for the one-hour head start of the first train.
Todo #4: Solve the resulting equation for the time they meet.

Let $D$ be the total distance (estimated as 200 miles). Let $t$ be the time in hours elapsed since the second train
left New York (3:00 pm).
\begin{itemize}
    \item Train from Boston ($\text{B}$): Speed $r_B = 60 \text{ mph}$. Leaves at 2:00 pm.
    \item Train from New York ($\text{N}$): Speed $r_N = 80 \text{ mph}$. Leaves at 3:00 pm.
\end{itemize}
Let the position of Boston be $x=0$ and New York be $x=D$.
The position of the Boston train at time $t$ (hours after 3:00 pm) is:
$x_B(t) = (\text{Initial distance covered}) + r_B \times t$
Since the Boston train started 1 hour earlier, its initial position at $t=0$ (3:00 pm) is $60 \text{ mph} \times 1 
\text{ hour} = 60$ miles from Boston at 2:00 pm, but since we measure time $t$ from 3:00 pm, we must use its 
position *at* 3:00 pm.
At 3:00 pm, the Boston train is $60 \text{ mph} \times 1 \text{ hour} = 60$ miles from Boston ($x=0$). From then 
on, its position is:
$x_B(t) = 60 + 60t$

The position of the New York train at time $t$ (hours after 3:00 pm) is (decreasing from $D$):
$x_N(t) = D - 80t$

They meet when $x_B(t) = x_N(t)$:
$60 + 60t = D - 80t$
$60 + 80t + 60t = D$
$140t = D - 60$
$t = \frac{D - 60}{140}$

Using $D = 200$ miles:
$t = \frac{200 - 60}{140} = \frac{140}{140} = 1$ hour.

Todo #1: Identify the distance between Boston and New York. Since it is not given, I must estimate a reasonable 
value for this distance.
Todo #2: Set up equations describing the position of each train as a function of time.
Todo #3: Account for the one-hour head start of the first train.
Todo #4: Solve the resulting equation for the time they meet.

The setup in Todo #2 already accounted for the head start by calculating the train's position at the starting time 
for the shared variable $t$.
The equations used were:
$x_B(t) = 60 + 60t$ (Position $t$ hours after 3:00 pm)
$x_N(t) = 200 - 80t$ (Position $t$ hours after 3:00 pm)
This step is conceptually merged with the solution in Todo #2, and the result was $t=1$ hour.

Todo #1: Identify the distance between Boston and New York. Since it is not given, I must estimate a reasonable 
value for this distance.
Todo #2: Set up equations describing the position of each train as a function of time.
Todo #3: Account for the one-hour head start of the first train.
Todo #4: Solve the resulting equation for the time they meet.

The time $t$ calculated in Todo #2 is the time elapsed *after* 3:00 pm.
$t = 1$ hour.
Meeting time = 3:00 pm + 1 hour = 4:00 pm.

Check the meeting point:
$x_B(1) = 60 + 60(1) = 120$ miles from Boston.
$x_N(1) = 200 - 80(1) = 120$ miles from Boston.
The positions match.

Todo #1: Identify the distance between Boston and New York. Since it is not given, I must estimate a reasonable 
value for this distance.
Todo #2: Set up equations describing the position of each train as a function of time.
Todo #3: Account for the one-hour head start of the first train.
Todo #4: Solve the resulting equation for the time they meet.

The trains meet at **4:00 pm**.

**Reasoning:**
1.  **Estimate Distance:** Since the distance between Boston and New York was not given, a reasonable estimate of 
**200 miles** was used.
2.  **Account for Head Start:** The train from Boston travels for 1 hour (from 2:00 pm to 3:00 pm) alone. In that 
time, it covers $60 \text{ mph} \times 1 \text{ hr} = 60$ miles.
3.  **Relative Speed Calculation (After 3:00 pm):** At 3:00 pm, the two trains are separated by $200 \text{ miles} 
- 60 \text{ miles} = 140$ miles. They are approaching each other at a combined relative speed of $60 \text{ mph} + 
80 \text{ mph} = 140 \text{ mph}$.
4.  **Time to Meet (After 3:00 pm):** Time = Distance / Speed = $140 \text{ miles} / 140 \text{ mph} = 1$ hour.
5.  **Meeting Time:** They meet 1 hour after 3:00 pm, which is 4:00 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>